In [0]:
#Load the Silver tables
df_gdp = spark.table("workspace.global_development.silver_gdp_growth")
df_population = spark.table("workspace.global_development.silver_population")
df_education = spark.table("workspace.global_development.silver_education")
df_country_metadata = spark.table("workspace.global_development.bronze_country_metadata")

df_gdp.printSchema()
df_population.printSchema()
df_education.printSchema()
df_country_metadata.printSchema()

root
 |-- Country_Code: string (nullable = true)
 |-- Country_Name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- gdp_growth_pct: double (nullable = true)
 |-- is_outlier: boolean (nullable = true)

root
 |-- Country_Code: string (nullable = true)
 |-- Country_Name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- population_total: double (nullable = true)
 |-- is_outlier: boolean (nullable = true)

root
 |-- Country_Code: string (nullable = true)
 |-- Country_Name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- education_expenditure_pct_gdp: double (nullable = true)
 |-- is_outlier: boolean (nullable = true)

root
 |-- Country_Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- IncomeGroup: string (nullable = true)



In [0]:
#Perform the 
from pyspark.sql import functions as F
df_gold_base = (
    df_population
    .withColumnRenamed("is_outlier", "population_is_outlier")
    .join(
        df_gdp.select("Country_Code", "year", "gdp_growth_pct", F.col("is_outlier").alias("gdp_is_outlier")),
        on=["Country_Code", "year"],
        how="left"
    )
    .join(
        df_education.select("Country_Code", "year", "education_expenditure_pct_gdp", F.col("is_outlier").alias("education_is_outlier")),
        on=["Country_Code", "year"],
        how="left"
    )
)

df_gold_base.printSchema()
df_gold_base.show(10)
print("Total rows:", df_gold_base.count())

root
 |-- Country_Code: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- Country_Name: string (nullable = true)
 |-- population_total: double (nullable = true)
 |-- population_is_outlier: boolean (nullable = true)
 |-- gdp_growth_pct: double (nullable = true)
 |-- gdp_is_outlier: boolean (nullable = true)
 |-- education_expenditure_pct_gdp: double (nullable = true)
 |-- education_is_outlier: boolean (nullable = true)

+------------+----+--------------------+----------------+---------------------+--------------+--------------+-----------------------------+--------------------+
|Country_Code|year|        Country_Name|population_total|population_is_outlier|gdp_growth_pct|gdp_is_outlier|education_expenditure_pct_gdp|education_is_outlier|
+------------+----+--------------------+----------------+---------------------+--------------+--------------+-----------------------------+--------------------+
|         ABW|1960|               Aruba|         54922.0|                fals

In [0]:
#Join in Region/IncomeGroup metadata - Broadcast on metadata
from pyspark.sql.functions import broadcast

df_gold_enriched = df_gold_base.join(
    broadcast(df_country_metadata),
    on="Country_Code",
    how="left"
)

df_gold_enriched.printSchema()
df_gold_enriched.show(5)
print("Total rows:", df_gold_enriched.count())

root
 |-- Country_Code: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- Country_Name: string (nullable = true)
 |-- population_total: double (nullable = true)
 |-- population_is_outlier: boolean (nullable = true)
 |-- gdp_growth_pct: double (nullable = true)
 |-- gdp_is_outlier: boolean (nullable = true)
 |-- education_expenditure_pct_gdp: double (nullable = true)
 |-- education_is_outlier: boolean (nullable = true)
 |-- Region: string (nullable = true)
 |-- IncomeGroup: string (nullable = true)

+------------+----+------------+----------------+---------------------+--------------+--------------+-----------------------------+--------------------+--------------------+-------------------+
|Country_Code|year|Country_Name|population_total|population_is_outlier|gdp_growth_pct|gdp_is_outlier|education_expenditure_pct_gdp|education_is_outlier|              Region|        IncomeGroup|
+------------+----+------------+----------------+---------------------+--------------+-----

In [0]:
#Add decade and population_growth_rate
from pyspark.sql import Window

country_year_window = Window.partitionBy("Country_Code").orderBy("year")

df_gold_final = (
    df_gold_enriched
    .withColumn("decade", (F.floor(F.col("year") / 10) * 10).cast("int"))
    .withColumn("prev_year_population", F.lag("population_total").over(country_year_window))
    .withColumn(
        "population_growth_rate",
        F.round(
            ((F.col("population_total") - F.col("prev_year_population")) / F.col("prev_year_population")) * 100,
            2
        )
    )
    .withColumn("gdp_growth_pct", F.round(F.col("gdp_growth_pct"), 2))
    .withColumn("education_expenditure_pct_gdp", F.round(F.col("education_expenditure_pct_gdp"), 2))
    .drop("prev_year_population")
)

df_gold_final.printSchema()

root
 |-- Country_Code: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- Country_Name: string (nullable = true)
 |-- population_total: double (nullable = true)
 |-- population_is_outlier: boolean (nullable = true)
 |-- gdp_growth_pct: double (nullable = true)
 |-- gdp_is_outlier: boolean (nullable = true)
 |-- education_expenditure_pct_gdp: double (nullable = true)
 |-- education_is_outlier: boolean (nullable = true)
 |-- Region: string (nullable = true)
 |-- IncomeGroup: string (nullable = true)
 |-- decade: integer (nullable = true)
 |-- population_growth_rate: double (nullable = true)



In [0]:
df_gold_final.select(
    "Country_Name", "year", "decade", "population_total", "population_growth_rate", "gdp_growth_pct"
).filter(F.col("Country_Name") == "Japan").orderBy("year").show(15)

+------------+----+------+----------------+----------------------+--------------+
|Country_Name|year|decade|population_total|population_growth_rate|gdp_growth_pct|
+------------+----+------+----------------+----------------------+--------------+
|       Japan|1960|  1960|        9.3216E7|                  NULL|          NULL|
|       Japan|1961|  1960|        9.4055E7|                   0.9|         12.04|
|       Japan|1962|  1960|        9.4933E7|                  0.93|          8.91|
|       Japan|1963|  1960|          9.59E7|                  1.02|          8.47|
|       Japan|1964|  1960|        9.6903E7|                  1.05|         11.68|
|       Japan|1965|  1960|        9.7952E7|                  1.08|          5.82|
|       Japan|1966|  1960|        9.8851E7|                  0.92|         10.64|
|       Japan|1967|  1960|        9.9879E7|                  1.04|         11.08|
|       Japan|1968|  1960|       1.01011E8|                  1.13|         12.88|
|       Japan|19

# gold_country_development_metrics
Provides a country-and-year-level view of GDP growth, population trends, 
and education investment (as % of GDP), enabling analysis of how these 
three factors relate to each other over time and across regions.

In [0]:
# Before writing Gold table - coalesce to avoid many small files (dataset is small here)
df_gold_final.coalesce(1).write.format("delta").mode("overwrite").saveAsTable(
    "workspace.global_development.gold_country_development_metrics")

print("Gold table created: gold_country_development_metrics")

In [0]:
display(spark.sql("SHOW TABLES IN workspace.global_development"))

database,tableName,isTemporary
global_development,bronze_country_metadata,false
global_development,bronze_education,false
global_development,bronze_gdp_growth,false
global_development,bronze_population,false
global_development,gold_country_development_metrics,false
global_development,silver_education,false
global_development,silver_education_with_nulls,false
global_development,silver_gdp_growth,false
global_development,silver_gdp_growth_with_nulls,false
global_development,silver_population,false


In [0]:
print("Total rows in Gold table:", spark.table("workspace.global_development.gold_country_development_metrics").count())

Total rows in Gold table: 14292


In [0]:
# Optimize the Gold table and ZORDER - small demo here only 
spark.sql("OPTIMIZE workspace.global_development.gold_country_development_metrics ZORDER BY (Country_Code, year)")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
# Permission for dataset and table from unity catalog level
#GRANT SELECT ON TABLE workspace.global_development.gold_country_development_metrics TO `some_user@example.com`;
#GRANT USE SCHEMA ON SCHEMA workspace.global_development TO `some_user@example.com`;